IMPORT IMPORTANT LIB

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
df = pd.read_csv("../dataset/Loan_default.csv")

REMOVING LOANID AGAIN BECOAUSE NEW FRESH NOTBOOK

In [3]:
df.drop("LoanID", axis=1, inplace=True)

In [4]:
X = df.drop("Default", axis=1)
y = df["Default"]

In [5]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

categorical_cols = X.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

FIRST MODEL (LOGISTIC REGRESSION)

In [8]:
log_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

In [9]:
log_model.fit(X_train, y_train)

c:\Users\DC\loan default prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


LogisticRegression(max_iter=1000, random_state=42)

In [10]:
y_pred = log_model.predict(X_test)

In [11]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.8850597219502644
Precision: 0.6020066889632107
Recall   : 0.030349013657056147
F1 Score : 0.05778491171749599


In [12]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[45020   119]
 [ 5751   180]]


In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.60      0.03      0.06      5931

    accuracy                           0.89     51070
   macro avg       0.74      0.51      0.50     51070
weighted avg       0.85      0.89      0.84     51070



SECOND EXPERIMENT - LOGISTIC REGRESSION WITH CLASS WEIGHT

In [14]:
log_balanced = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

In [15]:
log_balanced.fit(X_train, y_train)

c:\Users\DC\loan default prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [16]:
y_pred_balanced = log_balanced.predict(X_test)

In [17]:
accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
f1_balanced = f1_score(y_test, y_pred_balanced)

print("Balanced Logistic Regression")
print("Accuracy :", accuracy_balanced)
print("Precision:", precision_balanced)
print("Recall   :", recall_balanced)
print("F1 Score :", f1_balanced)

Balanced Logistic Regression
Accuracy : 0.673996475425886
Precision: 0.21880575086577816
Recall   : 0.7030854830551341
F1 Score : 0.3337468486133899


In [18]:
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print("Confusion Matrix:")
print(cm_balanced)

Confusion Matrix:
[[30251 14888]
 [ 1761  4170]]


In [19]:
print("Classification Report:")
print(classification_report(y_test, y_pred_balanced))

Classification Report:


              precision    recall  f1-score   support

           0       0.94      0.67      0.78     45139
           1       0.22      0.70      0.33      5931

    accuracy                           0.67     51070
   macro avg       0.58      0.69      0.56     51070
weighted avg       0.86      0.67      0.73     51070



 THIRD EXPERIMENT - THRESHOLD TUNING

In [20]:
# Get probability of Default (Class 1)
y_prob = log_balanced.predict_proba(X_test)[:, 1]

print(y_prob[:10])

[0.18915992 0.16625811 0.3404981  0.60253853 0.44787949 0.07104604
 0.80846827 0.34963673 0.80994562 0.54353805]


In [21]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)
    accuracy = accuracy_score(y_test, y_pred_threshold)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

threshold_results = pd.DataFrame(results)

threshold_results

,Threshold,Accuracy,Precision,Recall,F1 Score
0,0.30,0.397552,0.152157,0.915866,0.260959
1,0.35,0.470942,0.164610,0.872534,0.276968
2,0.40,0.545252,0.180717,0.825156,0.296498
3,0.45,0.611768,0.198097,0.768673,0.315011
4,0.50,0.673996,0.218806,0.703085,0.333747
5,0.55,0.728725,0.241905,0.626033,0.348966
6,0.60,0.775250,0.268199,0.541055,0.358628
7,0.65,0.814940,0.299133,0.441915,0.356769
8,0.70,0.844684,0.335309,0.343450,0.339330


In [22]:
best_threshold = 0.60

y_pred_threshold = (y_prob >= best_threshold).astype(int)

print("Threshold:", best_threshold)

print("Accuracy :", accuracy_score(y_test, y_pred_threshold))
print("Precision:", precision_score(y_test, y_pred_threshold))
print("Recall   :", recall_score(y_test, y_pred_threshold))
print("F1 Score :", f1_score(y_test, y_pred_threshold))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_threshold))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_threshold))

Threshold: 0.6
Accuracy : 0.7752496573330723
Precision: 0.26819891349770164
Recall   : 0.5410554712527399
F1 Score : 0.3586276262852034

Confusion Matrix:
[[36383  8756]
 [ 2722  3209]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.81      0.86     45139
           1       0.27      0.54      0.36      5931

    accuracy                           0.78     51070
   macro avg       0.60      0.67      0.61     51070
weighted avg       0.85      0.78      0.81     51070



 MODEL 2 - DECISION TREE

In [23]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

In [24]:
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [25]:
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [26]:
y_pred_dt = dt_model.predict(X_test)

In [27]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

print("Decision Tree")
print("Accuracy :", accuracy_dt)
print("Precision:", precision_dt)
print("Recall   :", recall_dt)
print("F1 Score :", f1_dt)

Decision Tree
Accuracy : 0.8177403563736049
Precision: 0.20465278992478572
Recall   : 0.19726858877086495
F1 Score : 0.20089285714285715


In [28]:
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("Confusion Matrix:")
print(cm_dt)

Confusion Matrix:
[[40592  4547]
 [ 4761  1170]]


In [29]:
print("Classification Report:")
print(classification_report(y_test, y_pred_dt))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.90      0.90     45139
           1       0.20      0.20      0.20      5931

    accuracy                           0.82     51070
   macro avg       0.55      0.55      0.55     51070
weighted avg       0.81      0.82      0.82     51070



MODEL 2B - TUNED DECISION TREE

In [30]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "DT_1": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=2,
        min_samples_leaf=1,
        criterion="gini",
        class_weight="balanced",
        random_state=42
    ),

    "DT_2": DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=1,
        criterion="gini",
        class_weight="balanced",
        random_state=42
    ),

    "DT_3": DecisionTreeClassifier(
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=1,
        criterion="gini",
        class_weight="balanced",
        random_state=42
    ),

    "DT_4": DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        criterion="gini",
        class_weight="balanced",
        random_state=42
    ),

    "DT_5": DecisionTreeClassifier(
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        criterion="entropy",
        class_weight="balanced",
        random_state=42
    )
}

In [31]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results)

print(results_df)

  Model  Accuracy  Precision    Recall        F1
0  DT_1  0.664715   0.205660  0.659248  0.313515
1  DT_2  0.666105   0.203012  0.640870  0.308348
2  DT_3  0.688212   0.192175  0.525881  0.281485
3  DT_4  0.665068   0.202661  0.642050  0.308078
4  DT_5  0.672313   0.190465  0.560445  0.284309


In [32]:

best_model_name = results_df.loc[
    results_df["F1"].idxmax(), "Model"
]

print("Best Model:", best_model_name)

Best Model: DT_1


In [33]:
best_dt = models[best_model_name]

y_pred_dt_tuned = best_dt.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred_dt_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt_tuned))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.67      0.78     45139
           1       0.21      0.66      0.31      5931

    accuracy                           0.66     51070
   macro avg       0.57      0.66      0.55     51070
weighted avg       0.85      0.66      0.72     51070

Confusion Matrix:
[[30037 15102]
 [ 2021  3910]]


MODEL 3 — BASIC RANDOM FOREST


In [34]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=1
)

In [35]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=5, min_samples_split=10,
                       n_estimators=50, n_jobs=1, random_state=42)

In [36]:
y_pred_rf = rf_model.predict(X_test)

In [37]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Random Forest")
print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)

Random Forest
Accuracy : 0.7287840219306834
Precision: 0.2429572893677788
Recall   : 0.6310908784353397
F1 Score : 0.3508459483526269


In [38]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Confusion Matrix:")
print(cm_rf)

Confusion Matrix:
[[33476 11663]
 [ 2188  3743]]


In [39]:
print("Classification Report:")
print(classification_report(y_test, y_pred_rf))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.74      0.83     45139
           1       0.24      0.63      0.35      5931

    accuracy                           0.73     51070
   macro avg       0.59      0.69      0.59     51070
weighted avg       0.86      0.73      0.77     51070



Model 3B — TUNED RANDOM FOREST 


In [40]:
rf_models = {
    "RF_1": RandomForestClassifier(
        n_estimators=50,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),

    "RF_2": RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),

    "RF_3": RandomForestClassifier(
        n_estimators=50,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),

    "RF_4": RandomForestClassifier(
        n_estimators=75,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),

    "RF_5": RandomForestClassifier(
        n_estimators=75,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    )
}

In [41]:
rf_results = []

for name, model in rf_models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    rf_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

rf_results_df = pd.DataFrame(rf_results)

print("\nRandom Forest Tuning Results:")
print(rf_results_df)

Training RF_1...
Training RF_2...
Training RF_3...
Training RF_4...
Training RF_5...

Random Forest Tuning Results:
  Model  Accuracy  Precision    Recall        F1
0  RF_1  0.702134   0.229811  0.665486  0.341643
1  RF_2  0.769042   0.264271  0.554207  0.357886
2  RF_3  0.828980   0.311702  0.391165  0.346942
3  RF_4  0.771627   0.266878  0.553195  0.360055
4  RF_5  0.831349   0.315289  0.385938  0.347055


In [42]:
best_rf_name = rf_results_df.loc[
    rf_results_df["F1"].idxmax(),
    "Model"
]

print("Best Random Forest:", best_rf_name)

Best Random Forest: RF_4


In [43]:
best_rf = rf_models[best_rf_name]

y_pred_rf_tuned = best_rf.predict(X_test)

In [44]:
print("Classification Report:")
print(classification_report(y_test, y_pred_rf_tuned))

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.80      0.86     45139
           1       0.27      0.55      0.36      5931

    accuracy                           0.77     51070
   macro avg       0.60      0.68      0.61     51070
weighted avg       0.85      0.77      0.80     51070



In [45]:
cm_rf_tuned = confusion_matrix(y_test, y_pred_rf_tuned)

print("Confusion Matrix:")
print(cm_rf_tuned)

Confusion Matrix:
[[36126  9013]
 [ 2650  3281]]


In [46]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [47]:
!pip install xgboost

In [48]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


MODEL 4 - XGBOOST

In [49]:
import xgboost as xgb

In [50]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=7.61,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=1
)

In [51]:
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=1,
              num_parallel_tree=None, ...)

In [52]:
print(xgb_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=1,
              num_parallel_tree=None, ...)


In [53]:
y_pred_xgb = xgb_model.predict(X_test)

In [54]:


print(y_pred_xgb[:10])

[0 0 0 1 0 0 1 0 1 1]


In [55]:
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print("XGBoost")
print("Accuracy :", accuracy_xgb)
print("Precision:", precision_xgb)
print("Recall   :", recall_xgb)
print("F1 Score :", f1_xgb)

XGBoost
Accuracy : 0.6903074211866066
Precision: 0.22646521666943384
Recall   : 0.689934243803743
F1 Score : 0.341


In [56]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("Confusion Matrix:")
print(cm_xgb)

Confusion Matrix:
[[31162 13977]
 [ 1839  4092]]


In [57]:
print("Classification Report:")
print(classification_report(y_test, y_pred_xgb))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.69      0.80     45139
           1       0.23      0.69      0.34      5931

    accuracy                           0.69     51070
   macro avg       0.59      0.69      0.57     51070
weighted avg       0.86      0.69      0.74     51070



In [58]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("Confusion Matrix:")
print(cm_xgb)

Confusion Matrix:
[[31162 13977]
 [ 1839  4092]]


MODEL 4B - TUNED XGBOOST

In [59]:
xgb_models = {
    "XGB_1": xgb.XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=7.61,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ),

    "XGB_2": xgb.XGBClassifier(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=7.61,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ),

    "XGB_3": xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=7.61,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ),

    "XGB_4": xgb.XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=7.61,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ),

    "XGB_5": xgb.XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=7.61,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    )
}

In [60]:
xgb_results = []

for name, model in xgb_models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    xgb_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

xgb_results_df = pd.DataFrame(xgb_results)

print("\nXGBoost Tuning Results:")
print(xgb_results_df)

Training XGB_1...


Training XGB_2...
Training XGB_3...
Training XGB_4...
Training XGB_5...

XGBoost Tuning Results:
   Model  Accuracy  Precision    Recall        F1
0  XGB_1  0.686626   0.224526  0.692126  0.339060
1  XGB_2  0.688584   0.226032  0.693644  0.340958
2  XGB_3  0.694909   0.228291  0.683527  0.342269
3  XGB_4  0.693186   0.227958  0.687911  0.342440
4  XGB_5  0.690210   0.226251  0.689091  0.340654


In [61]:
best_xgb_name = xgb_results_df.loc[
    xgb_results_df["F1"].idxmax(),
    "Model"
]

print("Best XGBoost:", best_xgb_name)

Best XGBoost: XGB_4


In [62]:
best_xgb = xgb_models[best_xgb_name]

y_pred_xgb_tuned = best_xgb.predict(X_test)

In [63]:
print("Classification Report:")
print(classification_report(y_test, y_pred_xgb_tuned))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.69      0.80     45139
           1       0.23      0.69      0.34      5931

    accuracy                           0.69     51070
   macro avg       0.59      0.69      0.57     51070
weighted avg       0.86      0.69      0.75     51070



In [64]:
cm_xgb_tuned = confusion_matrix(y_test, y_pred_xgb_tuned)

print("Confusion Matrix:")
print(cm_xgb_tuned)

Confusion Matrix:
[[31321 13818]
 [ 1851  4080]]


MODEL 5 THRESHOLD OPTIMIZATION FOR THE TUNED RANDOM FOREST 

In [65]:
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

print(y_prob_rf[:10])

[0.20624112 0.14614236 0.25316614 0.41196931 0.346434   0.0959234
 0.695603   0.18128968 0.66900386 0.48826176]


In [66]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred_threshold),
        "Precision": precision_score(y_test, y_pred_threshold),
        "Recall": recall_score(y_test, y_pred_threshold),
        "F1": f1_score(y_test, y_pred_threshold)
    })

threshold_results_df = pd.DataFrame(threshold_results)

print(threshold_results_df)

   Threshold  Accuracy  Precision    Recall        F1
0       0.30  0.478794   0.166285  0.868993  0.279153
1       0.35  0.569062   0.186572  0.806778  0.303059
2       0.40  0.648306   0.209953  0.734109  0.326521
3       0.45  0.715351   0.236303  0.650143  0.346622
4       0.50  0.771627   0.266878  0.553195  0.360055
5       0.55  0.813119   0.297772  0.448491  0.357912
6       0.60  0.844703   0.338240  0.352554  0.345249
7       0.65  0.866732   0.389492  0.259990  0.311830
8       0.70  0.879851   0.454139  0.171135  0.248592


In [67]:
best_threshold = threshold_results_df.loc[
    threshold_results_df["F1"].idxmax(),
    "Threshold"
]

print("Best Threshold:", best_threshold)

Best Threshold: 0.5


In [68]:
y_pred_rf_threshold = (
    y_prob_rf >= best_threshold
).astype(int)

In [69]:
print("Threshold:", best_threshold)

print("Accuracy :", accuracy_score(y_test, y_pred_rf_threshold))
print("Precision:", precision_score(y_test, y_pred_rf_threshold))
print("Recall   :", recall_score(y_test, y_pred_rf_threshold))
print("F1 Score :", f1_score(y_test, y_pred_rf_threshold))

Threshold: 0.5
Accuracy : 0.7716271783826121
Precision: 0.26687815194403774
Recall   : 0.5531950767155623
F1 Score : 0.3600548696844993


In [70]:
print("Classification Report:")
print(classification_report(y_test, y_pred_rf_threshold))

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.80      0.86     45139
           1       0.27      0.55      0.36      5931

    accuracy                           0.77     51070
   macro avg       0.60      0.68      0.61     51070
weighted avg       0.85      0.77      0.80     51070



In [71]:
cm_rf_threshold = confusion_matrix(
    y_test,
    y_pred_rf_threshold
)

print("Confusion Matrix:")
print(cm_rf_threshold)

Confusion Matrix:
[[36126  9013]
 [ 2650  3281]]


In [72]:
from sklearn.metrics import roc_auc_score, average_precision_score

In [73]:
# Logistic Regression
y_prob_lr = log_balanced.predict_proba(X_test)[:, 1]

# Tuned Random Forest
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

# Tuned XGBoost
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

In [74]:
model_auc_results = []

model_auc_results.append({
    "Model": "Balanced Logistic Regression",
    "ROC-AUC": roc_auc_score(y_test, y_prob_lr),
    "PR-AUC": average_precision_score(y_test, y_prob_lr)
})

model_auc_results.append({
    "Model": "Tuned Random Forest",
    "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
    "PR-AUC": average_precision_score(y_test, y_prob_rf)
})

model_auc_results.append({
    "Model": "Tuned XGBoost",
    "ROC-AUC": roc_auc_score(y_test, y_prob_xgb),
    "PR-AUC": average_precision_score(y_test, y_prob_xgb)
})

auc_results_df = pd.DataFrame(model_auc_results)

print(auc_results_df)

                          Model   ROC-AUC    PR-AUC
0  Balanced Logistic Regression  0.749811  0.304628
1           Tuned Random Forest  0.750654  0.314666
2                 Tuned XGBoost  0.757881  0.331336


In [75]:
final_comparison = pd.DataFrame({
    "Model": [
        "Balanced Logistic Regression",
        "Balanced LR + Threshold 0.60",
        "Tuned Decision Tree",
        "Tuned Random Forest",
        "Tuned XGBoost"
    ],
    
    "Accuracy": [
        accuracy_score(y_test, y_pred_balanced),
        0.7752496573330723,
        accuracy_score(y_test, y_pred_dt_tuned),
        accuracy_score(y_test, y_pred_rf_tuned),
        accuracy_score(y_test, y_pred_xgb_tuned)
    ],
    
    "Precision": [
        precision_score(y_test, y_pred_balanced),
        0.26819891349770164,
        precision_score(y_test, y_pred_dt_tuned),
        precision_score(y_test, y_pred_rf_tuned),
        precision_score(y_test, y_pred_xgb_tuned)
    ],
    
    "Recall": [
        recall_score(y_test, y_pred_balanced),
        0.5410554712527399,
        recall_score(y_test, y_pred_dt_tuned),
        recall_score(y_test, y_pred_rf_tuned),
        recall_score(y_test, y_pred_xgb_tuned)
    ],
    
    "F1": [
        f1_score(y_test, y_pred_balanced),
        0.3586276262852034,
        f1_score(y_test, y_pred_dt_tuned),
        f1_score(y_test, y_pred_rf_tuned),
        f1_score(y_test, y_pred_xgb_tuned)
    ],
    
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, log_balanced.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, best_dt.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb)
    ],
    
    "PR-AUC": [
        average_precision_score(y_test, y_prob_lr),
        average_precision_score(
            y_test,
            log_balanced.predict_proba(X_test)[:, 1]
        ),
        average_precision_score(
            y_test,
            best_dt.predict_proba(X_test)[:, 1]
        ),
        average_precision_score(y_test, y_prob_rf),
        average_precision_score(y_test, y_prob_xgb)
    ]
})

print(final_comparison.round(4))

                          Model  Accuracy  Precision  Recall      F1  ROC-AUC  \
0  Balanced Logistic Regression    0.6740     0.2188  0.7031  0.3337   0.7498   
1  Balanced LR + Threshold 0.60    0.7752     0.2682  0.5411  0.3586   0.7498   
2           Tuned Decision Tree    0.6647     0.2057  0.6592  0.3135   0.7186   
3           Tuned Random Forest    0.7716     0.2669  0.5532  0.3601   0.7507   
4                 Tuned XGBoost    0.6932     0.2280  0.6879  0.3424   0.7579   

   PR-AUC  
0  0.3046  
1  0.3046  
2  0.2557  
3  0.3147  
4  0.3313  


FINAL PRODUCTION PIPELINE

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

import joblib

print("All imports successful!")

All imports successful!


In [3]:
df_final = pd.read_csv("../dataset/Loan_default.csv")

print("Dataset shape:", df_final.shape)
print(df_final.columns.tolist())

Dataset shape: (255347, 18)
['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default']


In [4]:
df_final = df_final.drop("LoanID", axis=1)

X_final = df_final.drop("Default", axis=1)
y_final = df_final["Default"]

print("Features shape:", X_final.shape)
print("Target shape:", y_final.shape)

Features shape: (255347, 16)
Target shape: (255347,)


In [5]:
categorical_cols = X_final.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X_final.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

Categorical columns:
['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']

Numerical columns:
['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']


In [6]:
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final,
    y_final,
    test_size=0.20,
    random_state=42,
    stratify=y_final
)

print("Training samples:", len(X_train_final))
print("Testing samples:", len(X_test_final))

Training samples: 204277
Testing samples: 51070


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_cols
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [8]:
final_rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=1
)

print(final_rf)

RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=5, min_samples_split=10,
                       n_estimators=50, n_jobs=1, random_state=42)


In [9]:
final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", final_rf)
    ]
)

print("Final pipeline created successfully.")

Final pipeline created successfully.


In [10]:
final_pipeline.fit(
    X_train_final,
    y_train_final
)

print("Final pipeline trained successfully.")

Final pipeline trained successfully.


In [11]:
y_pred_final = final_pipeline.predict(X_test_final)

print("Predictions generated successfully.")

Predictions generated successfully.


In [12]:
final_accuracy = accuracy_score(y_test_final, y_pred_final)
final_precision = precision_score(y_test_final, y_pred_final)
final_recall = recall_score(y_test_final, y_pred_final)
final_f1 = f1_score(y_test_final, y_pred_final)

print("FINAL RANDOM FOREST MODEL")
print("-------------------------")
print("Accuracy :", final_accuracy)
print("Precision:", final_precision)
print("Recall   :", final_recall)
print("F1 Score :", final_f1)

FINAL RANDOM FOREST MODEL
-------------------------
Accuracy : 0.7713138829058156
Precision: 0.26473477406679763
Recall   : 0.5452706120384421
F1 Score : 0.3564225491816829


In [13]:
print("Classification Report:")
print(classification_report(y_test_final, y_pred_final))

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.80      0.86     45139
           1       0.26      0.55      0.36      5931

    accuracy                           0.77     51070
   macro avg       0.60      0.67      0.61     51070
weighted avg       0.85      0.77      0.80     51070



In [14]:
final_cm = confusion_matrix(y_test_final, y_pred_final)

print("Confusion Matrix:")
print(final_cm)

Confusion Matrix:
[[36157  8982]
 [ 2697  3234]]


In [15]:
y_prob_final = final_pipeline.predict_proba(X_test_final)[:, 1]

final_roc_auc = roc_auc_score(
    y_test_final,
    y_prob_final
)

final_pr_auc = average_precision_score(
    y_test_final,
    y_prob_final
)

print("ROC-AUC:", final_roc_auc)
print("PR-AUC :", final_pr_auc)

ROC-AUC: 0.7505066582602533
PR-AUC : 0.3116845140904559


SAVE THE FINAL MODEL 

In [16]:
joblib.dump(
    final_pipeline,
    "../ml/final_loan_default_pipeline.pkl"
)

print("Final model saved successfully!")

Final model saved successfully!


In [2]:
import os

model_path = "../ml/final_loan_default_pipeline.pkl"

print("File exists:", os.path.exists(model_path))
print("File path:", model_path)

File exists: True
File path: ../ml/final_loan_default_pipeline.pkl


In [4]:
import joblib
loaded_pipeline = joblib.load(
    "../ml/final_loan_default_pipeline.pkl"
)

print("Saved model loaded successfully!")

Saved model loaded successfully!


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df_test = pd.read_csv("../dataset/Loan_default.csv")

# Remove ID
df_test = df_test.drop("LoanID", axis=1)

# Separate features and target
X = df_test.drop("Default", axis=1)
y = df_test["Default"]

# Recreate the same test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Test data recreated successfully!")
print("Test samples:", len(X_test_final))

Test data recreated successfully!
Test samples: 51070


In [7]:
test_prediction = loaded_pipeline.predict(
    X_test_final.iloc[:5]
)

test_probability = loaded_pipeline.predict_proba(
    X_test_final.iloc[:5]
)[:, 1]

print("Predictions:", test_prediction)
print("Default probabilities:", test_probability)

Predictions: [0 0 0 0 0]
Default probabilities: [0.18129761 0.17229553 0.30443067 0.3849749  0.36986641]


LOAN RISK PREDICTION FUNCTION

In [11]:
def predict_loan_risk(applicant_data):

    # Convert applicant data into DataFrame
    applicant_df = pd.DataFrame([applicant_data])

    # Get prediction
    prediction = loaded_pipeline.predict(applicant_df)[0]

    # Get probability of default
    probability = loaded_pipeline.predict_proba(
        applicant_df
    )[0][1]

    # Convert probability to percentage
    default_probability = probability * 100

    # Risk category
    if default_probability < 30:
        risk_level = "Low"
    elif default_probability < 60:
        risk_level = "Medium"
    else:
        risk_level = "High"

    return {
        "prediction": int(prediction),
        "default_probability": float(round(default_probability, 2)),
        "risk_level": risk_level
    }

In [9]:
sample_applicant = {
    "Age": 35,
    "Income": 50000,
    "LoanAmount": 200000,
    "CreditScore": 720,
    "MonthsEmployed": 60,
    "NumCreditLines": 3,
    "InterestRate": 10.5,
    "LoanTerm": 36,
    "DTIRatio": 0.30,
    "Education": "Bachelor's",
    "EmploymentType": "Full-time",
    "MaritalStatus": "Married",
    "HasMortgage": "Yes",
    "HasDependents": "No",
    "LoanPurpose": "Home",
    "HasCoSigner": "Yes"
}

In [12]:
result = predict_loan_risk(sample_applicant)

print(result)

{'prediction': 0, 'default_probability': 37.34, 'risk_level': 'Medium'}
